In [1]:
import os
# import pickle
# import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# from bvh_converter import bvh_mod
# from scipy.signal import savgol_filter

# from matplotlib.lines import Line2D
# from scipy.interpolate import interp1d

from utils_trajectory.trajectory_by_subdiv import plot_trajectories_by_subdiv
from utils_trajectory.trajectory_by_beat import plot_trajectories_by_beat
from utils_trajectory.trajectory_cycle import *

## A.1 Trajectory subdivision wise plots by dance modes

In [4]:
file_name = "BKO_E1_D1_01_Suku" #mode_csv.split("_Dancers")[0]
mode_df = pd.read_csv("data/subset_dance_annotation/" + file_name + "_Dancers.csv")

mode_group = mode_df[mode_df["mocap"] == "gr"].reset_index(drop=True)
mode_individual = mode_df[mode_df["mocap"] == "in"].reset_index(drop=True)
mode_audience = mode_df[mode_df["mocap"] == "au"].reset_index(drop=True)

# helper to extract all (start, end) tuples for a mode
def get_segments(df, name):
    if df.empty:
        # print(f"⚠️  No rows for mode '{name}', skipping.")
        return None
    return [(row["Start (in sec)"], row["End (in sec)"]) for _, row in df.iterrows()]

# build a dict of segments
segments = {
    "group":      get_segments(mode_group,      "gr"),
    "individual": get_segments(mode_individual, "in"),
    "audience":   get_segments(mode_audience,   "au")
}

def get_tsegment_for(mode_name, mode_value, suffix):
    """
    Run get_segments on one mode, and return a one-entry dict
    iff it isn’t None.
    """
    seg = get_segments(mode_value, suffix)
    return mode_name, seg if seg is not None else {}


mode_name, segmnt = get_tsegment_for("group", mode_group,      "gr")            # comment uncomment
# mode_name, segmnt = get_tsegment_for("individual", mode_individual, "in")     # comment uncomment
# mode_name, segmnt   = get_tsegment_for("audience",   mode_audience,   "au")   # comment uncomment
    

fig, ax = plot_trajectories_by_subdiv(
    file_name=file_name,
    mode=mode_name,
    base_path_cycles="data/virtual_cycles",
    base_path_logs="data/dance_onsets_v4_0.007_foot_jun3",
    output_dir=f"output_trajectory_plot/{mode_name}",
    time_segments=segmnt,  # Pass all segments for this mode
    n_beats_per_cycle=4, 
    n_subdiv_per_beat=3, 
    nn=2,       # number of subdiv you want around 0
    use_cycles=True,
    show_gray_plots=True,
    subdiv_set= [2,5,8,11],      #   [1,4,7,10]   [2,5,8,11] [3,6,9,12]
    show_trajectories= True,  # New parameter to control trajectory lines
    show_vlines= False        # New parameter to control vertical lines
)

plt.show(fig)

# fig.savefig(os.path.join(save_dir, f"{file_name}_{mode}.png"))
# plt.close(fig)

### Batch Export: Trajectory subdivision plots by dance modes

In [ ]:
mode_csv_list = os.listdir("data/subset_dance_annotation")

for mode_csv in mode_csv_list:
    file_name = mode_csv.split("_Dancers")[0]
    mode_df = pd.read_csv("data/subset_dance_annotation/" + mode_csv)

    mode_group = mode_df[mode_df["mocap"] == "gr"].reset_index(drop=True)
    mode_individual = mode_df[mode_df["mocap"] == "in"].reset_index(drop=True)
    mode_audience = mode_df[mode_df["mocap"] == "au"].reset_index(drop=True)

    # helper to extract all (start, end) tuples for a mode
    def get_segments(df, name):
        if df.empty:
            print(f"⚠️  No rows for mode '{name}', skipping.")
            return None
        return [(row["Start (in sec)"], row["End (in sec)"]) for _, row in df.iterrows()]

    # build a dict of segments
    segments = {
        "group":      get_segments(mode_group,      "gr"),
        "individual": get_segments(mode_individual, "in"),
        "audience":   get_segments(mode_audience,   "au")
    }

    # filter out the empty ones
    tsegment = {mode: seg for mode, seg in segments.items() if seg is not None}

    # save_dir = f"output_static_plot/foot_trajectories_subdiv/{file_name}"
    # os.makedirs(save_dir, exist_ok=True)
    for mode, segments in tsegment.items():
        fig, ax = plot_trajectories_by_subdiv(
            file_name=file_name,
            mode=mode,
            base_path_cycles="data/virtual_cycles",
            base_path_logs="data/dance_onsets_v4_0.007_foot_jun3",
            output_dir=f"output_trajectory_plot/{mode}",
            time_segments=segments,  # Pass all segments for this mode
            n_beats_per_cycle=4, 
            n_subdiv_per_beat=3, 
            nn=2,           # number of subdiv you want around 0
            use_cycles=True,
            show_gray_plots=True,
            subdiv_set=[2,5,8,11],
            show_trajectories=True,  # New parameter to control trajectory lines
            show_vlines=False        # New parameter to control vertical lines
        )
        # Create a filename that includes all segments
        # segment_str = "_".join([f"{start:.1f}_{end:.1f}" for start, end in segments])
        # fig.legend()
        # fig.savefig(os.path.join(save_dir, f"{file_name}_{mode}.png"), bbox_inches="tight")
        # plt.close(fig)

## A.2 Trajectory beat wise plots by dance modes

In [5]:
file_name =  "BKO_E2_D3_06_Manjanin"           # "BKO_E1_D1_01_Suku" #mode_csv.split("_Dancers")[0]
mode_df = pd.read_csv("data/subset_dance_annotation/"  + file_name + "_Dancers.csv")

mode_group = mode_df[mode_df["mocap"] == "gr"].reset_index(drop=True)
mode_individual = mode_df[mode_df["mocap"] == "in"].reset_index(drop=True)
mode_audience = mode_df[mode_df["mocap"] == "au"].reset_index(drop=True)

# helper to extract all (start, end) tuples for a mode
def get_segments(df, name):
    if df.empty:
        # print(f"⚠️  No rows for mode '{name}', skipping.")
        return None
    return [(row["Start (in sec)"], row["End (in sec)"]) for _, row in df.iterrows()]

# build a dict of segments
segments = {
    "group":      get_segments(mode_group,      "gr"),
    "individual": get_segments(mode_individual, "in"),
    "audience":   get_segments(mode_audience,   "au")
}

def get_tsegment_for(mode_name, mode_value, suffix):
    """
    Run get_segments on one mode, and return a one-entry dict
    iff it isn’t None.
    """
    seg = get_segments(mode_value, suffix)
    return mode_name, seg if seg is not None else {}


mode_name, segmnt = get_tsegment_for("group", mode_group,      "gr")
# mode_name, segmnt = get_tsegment_for("individual", mode_individual, "in")
# mode_name, segmnt   = get_tsegment_for("audience",   mode_audience,   "au")
    
    # filter out the empty ones
    # tsegment = {mode: seg for mode, seg in segments.items() if seg is not None}


# save_dir = f"output_static_plot/foot_trajectories_beat/{file_name}"
# os.makedirs(save_dir, exist_ok=True)

fig, ax = plot_trajectories_by_beat(
            file_name=file_name,
            mode=mode_name,
            base_path_cycles="data/virtual_cycles",
            base_path_logs= "data/dance_onsets_v4_0.007_foot_jun3",  #        "data/logs_v3_0.2__lower_jun3",                 #"data/logs_v1_may",
            output_dir=f"output_trajectory_plot/{mode_name}",
            time_segments=segmnt,  # Pass all segments for this mode
            n_beats_per_cycle=4, 
            n_subdiv_per_beat=3, 
            nn=3,
            use_cycles=True,
            show_gray_plots=True,
            show_trajectories=True,  # New parameter to control trajectory lines
            show_vlines=False        # New parameter to control vertical lines
        )

plt.show(fig)


# fig.savefig(os.path.join(save_dir, f"{file_name}_{mode}.png"))
# plt.close(fig)

### Batch Export: Trajectory beat wise plots by dance modes

In [ ]:
mode_csv_list = os.listdir("data/subset_dance_annotation")

for mode_csv in mode_csv_list:
    file_name = mode_csv.split("_Dancers")[0]
    mode_df = pd.read_csv("data/subset_dance_annotation/" + mode_csv)

    mode_group = mode_df[mode_df["mocap"] == "gr"].reset_index(drop=True)
    mode_individual = mode_df[mode_df["mocap"] == "in"].reset_index(drop=True)
    mode_audience = mode_df[mode_df["mocap"] == "au"].reset_index(drop=True)

    # helper to extract all (start, end) tuples for a mode
    def get_segments(df, name):
        if df.empty:
            print(f"⚠️  No rows for mode '{name}', skipping.")
            return None
        return [(row["Start (in sec)"], row["End (in sec)"]) for _, row in df.iterrows()]

    # build a dict of segments
    segments = {
        "group":      get_segments(mode_group,      "gr"),
        "individual": get_segments(mode_individual, "in"),
        "audience":   get_segments(mode_audience,   "au")
    }

    # filter out the empty ones
    tsegment = {mode: seg for mode, seg in segments.items() if seg is not None}

    for mode, segments in tsegment.items():
        fig, ax = plot_trajectories_by_beat(
            file_name=file_name,
            mode=mode,
            base_path_cycles="data/virtual_cycles",
            base_path_logs= "data/dance_onsets_v4_0.007_foot_jun3",          # "data/logs_v2_may",   # "data/logs_v3_0.2__lower_jun3",    # "data/logs_v4_0.007_foot_jun3", 
            output_dir=f"output_trajectory_plot/{mode_name}",
            time_segments=segments,  # Pass all segments for this mode
            n_beats_per_cycle=4, 
            n_subdiv_per_beat=3, 
            nn=2,
            use_cycles=True,
            show_gray_plots=True,
            show_trajectories=True,  # New parameter to control trajectory lines
            show_vlines=False        # New parameter to control vertical lines
        )


# Not sure below -- maybe discard

### Trajectory plot by cycle with average (without thresholding)

In [ ]:
# mode_csv_list = os.listdir("data/subset_dance_annotation")

# for mode_csv in mode_csv_list:
file_name = "BKO_E1_D1_02_Maraka" #mode_csv.split("_Dancers")[0]
mode_df = pd.read_csv("data/subset_dance_annotation/" + file_name + "_Dancers.csv")

mode_group = mode_df[mode_df["mocap"] == "gr"].reset_index(drop=True)
mode_individual = mode_df[mode_df["mocap"] == "in"].reset_index(drop=True)
mode_audience = mode_df[mode_df["mocap"] == "au"].reset_index(drop=True)

# helper to extract all (start, end) tuples for a mode
def get_segments(df, name):
    if df.empty:
        # print(f"⚠️  No rows for mode '{name}', skipping.")
        return None
    return [(row["Start (in sec)"], row["End (in sec)"]) for _, row in df.iterrows()]

# build a dict of segments
segments = {
    "group":      get_segments(mode_group,      "gr"),
    "individual": get_segments(mode_individual, "in"),
    "audience":   get_segments(mode_audience,   "au")
}

def get_tsegment_for(mode_name, mode_value, suffix):
    """
    Run get_segments on one mode, and return a one-entry dict
    iff it isn’t None.
    """
    seg = get_segments(mode_value, suffix)
    return mode_name, seg if seg is not None else {}


# mode_name, segmnt = get_tsegment_for("group", mode_group,      "gr")
# mode_name, segmnt = get_tsegment_for("individual", mode_individual, "in")
mode_name, segmnt   = get_tsegment_for("audience",   mode_audience,   "au")

# save_dir = f"output_static_plot/hands_cycles_trajectories/{file_name}"
# os.makedirs(save_dir, exist_ok=True)


fig, ax = plot_hand_norms_trajectories(        # change function name for feet or hands
            file_name=file_name,
            mode=mode_name,
            base_path_cycles="data/virtual_cycles",
            time_segments=segmnt,  # Pass all segments for this mode
            select_axis = 1,
            figsize = (12, 11),
            n_beats_per_cycle=4, 
            n_subdiv_per_beat=3, 
            show_gray_plots= True,
            show_trajectories= True,  # New parameter to control trajectory lines
            show_vlines= False        # New parameter to control vertical lines
        )


plt.show(fig)

# fig.savefig(os.path.join(save_dir, f"{file_name}_{mode}.png"))
# plt.close(fig)

#### Process all: Trajectory plot by cycle

In [ ]:
save_dir = f"output_static_plot/hand_norms_trajectories_rel/"
os.makedirs(save_dir, exist_ok=True)


piece_name = ['BKO_E1_D1_01_Suku','BKO_E1_D1_01_Suku','BKO_E1_D1_01_Suku','BKO_E1_D1_01_Suku','BKO_E1_D1_02_Maraka','BKO_E1_D1_02_Maraka','BKO_E1_D1_02_Maraka','BKO_E1_D1_02_Maraka','BKO_E1_D1_03_Wasulunka','BKO_E1_D1_06_Manjanin','BKO_E1_D2_03_Suku','BKO_E1_D2_05_Wasulunka','BKO_E2_D3_01_Maraka','BKO_E2_D3_01_Maraka','BKO_E2_D3_01_Maraka','BKO_E2_D3_01_Maraka','BKO_E2_D3_01_Maraka','BKO_E2_D3_02_Suku','BKO_E2_D3_02_Suku','BKO_E2_D3_02_Suku','BKO_E2_D3_02_Suku','BKO_E2_D3_02_Suku','BKO_E2_D3_02_Suku','BKO_E2_D3_02_Suku','BKO_E2_D3_03_Wasulunka','BKO_E2_D3_03_Wasulunka','BKO_E2_D3_06_Manjanin','BKO_E2_D3_06_Manjanin','BKO_E2_D3_06_Manjanin','BKO_E2_D3_06_Manjanin','BKO_E2_D3_14_Maraka','BKO_E2_D3_14_Maraka','BKO_E2_D4_01_Suku','BKO_E2_D4_01_Suku','BKO_E2_D4_01_Suku','BKO_E2_D4_01_Suku','BKO_E2_D4_02_Maraka','BKO_E2_D4_02_Maraka','BKO_E2_D4_03_Wasulunka','BKO_E2_D4_03_Wasulunka','BKO_E2_D4_03_Wasulunka','BKO_E2_D4_06_Manjanin','BKO_E2_D4_06_Manjanin','BKO_E2_D4_06_Manjanin','BKO_E2_D4_06_Manjanin','BKO_E2_D4_06_Manjanin','BKO_E2_D4_12_Suku','BKO_E2_D4_12_Suku','BKO_E2_D4_12_Suku','BKO_E2_D4_12_Suku','BKO_E3_D5_01_Maraka','BKO_E3_D5_02_Suku','BKO_E3_D5_02_Suku','BKO_E3_D5_03_Wasulunka','BKO_E3_D5_03_Wasulunka','BKO_E3_D5_03_Wasulunka','BKO_E3_D5_06_Manjanin','BKO_E3_D5_06_Manjanin','BKO_E3_D5_06_Manjanin','BKO_E3_D5_06_Manjanin','BKO_E3_D5_13_Suku','BKO_E3_D5_13_Suku','BKO_E3_D6_01_Maraka','BKO_E3_D6_01_Maraka','BKO_E3_D6_01_Maraka','BKO_E3_D6_02_Suku','BKO_E3_D6_02_Suku','BKO_E3_D6_02_Suku','BKO_E3_D6_06_Manjanin','BKO_E3_D6_12_Suku','BKO_E3_D6_12_Suku']
time_segments = [ (92.085,116.46), (112.76,143.14), (178.1,198.288), (195.16,221.579), (65.2,137.78), (137.78,156.18), (155.46,168.58), (205.9,268.06), (241.9,288.16), (146.28,186.74), (113.4,153.54), (78.8,123.42), (81.8,92.32), (123.16,145.3), (160.1,169.74), (223.28,236.52), (224.22,238.44), (116.28,130.52), (171.86,186.96), (185.74,214.44), (187.74,216.36), (233.12,250.68), (233.74,250.84), (250.46,265.58), (167.2,185.48), (167.22,185.44), (86.34,101.24), (99.46,118.86), (178.42,191.32), (192.36,246.76), (73.42,93.74), (91.38,101.46), (145.1,155.46), (158.0,172.88), (206.98,230.38), (232.42,250.64), (94.9,106.88), (121.42,133.84), (88.94,103.68), (129.62,139.0), (160.54,185.76), (99.36,116.24), (137.06,150.9), (148.08,164.52), (169.82,191.06), (169.84,185.66), (72.34,84.3), (72.62,85.92), (85.42,98.32), (115.16,137.04), (70.98,93.5), (68.02,86.42), (193.74,210.76), (128.24,148.1), (149.16,195.1), (171.18,186.28), (125.64,182.0), (211.26,261.12), (259.58,280.0), (279.0,316.58), (71.92,91.72), (120.88,162.72), (51.42,70.02), (69.7,101.7), (123.1,152.0), (70.78,90.34), (90.34,114.86), (181.28,200.06), (286.56,307.74), (159.46,179.44), (249.24,271.08) ]


for filename, segment in zip(piece_name, time_segments):
    os.makedirs(save_dir, exist_ok=True)


    fig, ax = plot_rel_hand_norms_trajectories(        # change function name for feet or hands
                file_name=filename,
                mode="audience",
                base_path_cycles="data/virtual_cycles",
                time_segments=[segment],  # Pass all segments for this mode
                select_axis = None,
                figsize = (12, 12),
                n_beats_per_cycle=4, 
                n_subdiv_per_beat=3, 
                show_trajectories= True,  # New parameter to control trajectory lines
                
                max_min_sec=0.35,    # 
                min_thresh=0.1, 
                slope_min=0.8       # slope ot 1 is 45 degree
            )


    # output_path = os.path.join(save_dir, f"{file_name}_{int(segment[0])}_{int(segment[1])}.png")

    # fig.savefig(output_path, bbox_inches="tight", dpi=300)

    fig.savefig(os.path.join(save_dir, f"{filename}_{segment[0]:.1f}_{segment[1]:.1f}.png"), bbox_inches="tight", dpi=300)
    plt.close(fig) 

In [ ]:
mode_csv_list = os.listdir("data/subset_dance_annotation")

for mode_csv in mode_csv_list:
    file_name = mode_csv.split("_Dancers")[0]
    mode_df = pd.read_csv("data/subset_dance_annotation/" + mode_csv)

    mode_group = mode_df[mode_df["mocap"] == "gr"].reset_index(drop=True)
    mode_individual = mode_df[mode_df["mocap"] == "in"].reset_index(drop=True)
    mode_audience = mode_df[mode_df["mocap"] == "au"].reset_index(drop=True)
    
    # helper to extract all (start, end) tuples for a mode
    def get_segments(df, name):
        if df.empty:
            print(f"⚠️  No rows for mode '{name}', skipping.")
            return None
        return [(row["Start (in sec)"], row["End (in sec)"]) for _, row in df.iterrows()]

    # build a dict of segments
    segments = {
        "group":      get_segments(mode_group,      "gr"),
        "individual": get_segments(mode_individual, "in"),
        "audience":   get_segments(mode_audience,   "au")
    }

    # filter out the empty ones
    tsegment = {mode: seg for mode, seg in segments.items() if seg is not None}

    segment_dir = f"output_static_plot/hand_norm_trajectories"
    os.makedirs(segment_dir, exist_ok=True)
    for mode, segments in tsegment.items():
        # Create subdirectory for this mode
        mode_dir = os.path.join(segment_dir, mode)
        os.makedirs(mode_dir, exist_ok=True)
        
        fig, ax = plot_hand_norms_trajectories(
                    file_name=file_name,
                    mode=mode,
                    base_path_cycles="data/virtual_cycles",
                    time_segments=segments,  # Pass all segments for this mode
                    select_axis = 1,    # 0: x(forward), 1: y(lateral), 2: z(vertical)
                    figsize = (12, 11),
                    n_beats_per_cycle=4, 
                    n_subdiv_per_beat=3, 
                    show_gray_plots= True,
                    show_trajectories= True,  # New parameter to control trajectory lines
                    show_vlines= False        # New parameter to control vertical lines
                )

        # plt.show(fig)
        save_dir = os.path.join(mode_dir, f"{file_name}_{mode}.png")

        fig.savefig(save_dir, bbox_inches="tight", dpi=300)
        plt.close(fig)

### Utils

### Move the plots into mode wise folder

In [ ]:
import os
import shutil

# Define your source and destination base directories
input_base_dir  = "output_static_plot/foot_trajectories_fullcycle"
output_base_dir = "output_static_plot/categorized_fullcycle"  # change this to wherever you want the three dirs

# List of categories and their corresponding suffixes
categories = ["audience", "group", "individual"]

# Create the category directories under the output base
for cat in categories:
    os.makedirs(os.path.join(output_base_dir, cat), exist_ok=True)

# Walk through each trajectory folder in the input base
for trajec in os.listdir(input_base_dir):
    trajec_path = os.path.join(input_base_dir, trajec)
    if not os.path.isdir(trajec_path):
        continue

    # Copy files matching each category into its directory
    for fname in os.listdir(trajec_path):
        for cat in categories:
            if fname.endswith(f"_{cat}.png"):
                src = os.path.join(trajec_path, fname)
                dst = os.path.join(output_base_dir, cat, fname)
                shutil.copy2(src, dst)
                break


In [ ]:
## mode time segments to pickle

import pickle

mode_csv_list = os.listdir("data/subset_dance_annotation")
for mode_csv in mode_csv_list:
    file_name = mode_csv.split("_Dancers")[0]
    mode_df = pd.read_csv("data/subset_dance_annotation/" + mode_csv)

    mode_group = mode_df[mode_df["mocap"] == "gr"].reset_index(drop=True)
    mode_individual = mode_df[mode_df["mocap"] == "in"].reset_index(drop=True)
    mode_audience = mode_df[mode_df["mocap"] == "au"].reset_index(drop=True)

    # helper to extract all (start, end) tuples for a mode
    def get_segments(df, name):
        if df.empty:
            print(f"⚠️  No rows for mode '{name}', skipping.")
            return None
        return [(row["Start (in sec)"], row["End (in sec)"]) for _, row in df.iterrows()]

    # build a dict of segments
    segments = {
        "group":      get_segments(mode_group,      "gr"),
        "individual": get_segments(mode_individual, "in"),
        "audience":   get_segments(mode_audience,   "au")
    }

    # filter out the empty ones
    tsegment = {mode: seg for mode, seg in segments.items() if seg is not None}

    output_name = mode_csv.split(".csv")[0]
    output_path = f"data/subset_dance_annotation_pickles/{output_name}.pkl"
    with open(output_path, "wb") as f:
        pickle.dump(tsegment, f)
 